# 🔍 Notebook 02 — Emerging Roles & Skills Analysis
> **Market Demand Trend Analysis** | *Job Role & Skill Intelligence*

**Objectives**
- Parse and explode skill lists from `job_skills.csv`
- Rank skills by demand frequency globally and by role
- Build a role × skill co-occurrence heatmap
- Identify outsourcing-relevant roles (Remote vs Onsite)
- Generate WordCloud of skills
- Cluster job titles using TF-IDF + KMeans

---

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from wordcloud import WordCloud, STOPWORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from utils import (
    set_style, load_merged, add_role_category, explode_skills,
    savefig, save_plotly, PALETTE, PLOTLY_TEMPLATE, OUT_REP
)

set_style()
print('✅ Setup complete')

## 1. Load Clean Data

In [ ]:
clean_path = OUT_REP / 'merged_clean.parquet'
if clean_path.exists():
    df = pd.read_parquet(clean_path)
    print(f'Loaded from parquet: {df.shape}')
else:
    df = load_merged()
    df = add_role_category(df)
    print(f'Loaded fresh: {df.shape}')

df.head(3)

## 2. Skill Frequency Analysis

In [ ]:
skills_exploded = explode_skills(df, skill_col='job_skills')
skill_freq      = skills_exploded['skill'].value_counts().reset_index()
skill_freq.columns = ['skill', 'count']

print(f'Unique skills: {len(skill_freq)}')
print('\nTop 20 skills:')
skill_freq.head(20)

In [ ]:
top_n  = 30
top_sk = skill_freq.head(top_n)

colors = plt.cm.plasma(np.linspace(0.2, 0.9, top_n))

fig, ax = plt.subplots(figsize=(10, 12))
bars = ax.barh(top_sk['skill'][::-1], top_sk['count'][::-1],
               color=colors, alpha=0.9, edgecolor='none', height=0.75)

for bar, val in zip(bars, top_sk['count'][::-1]):
    ax.text(bar.get_width() + 4, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=8.5, color='#DEDEDE')

ax.set_title(f'Top {top_n} In-Demand Skills — Jan 2024', fontweight='bold', pad=15)
ax.set_xlabel('Number of Job Postings Requiring This Skill')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
savefig(fig, '02_top_skills_frequency')
plt.show()

## 3. Skills WordCloud

In [ ]:
skill_text = ' '.join(skills_exploded['skill'].tolist())

custom_stopwords = set(STOPWORDS) | {'experience', 'years', 'skills', 'ability',
                                      'knowledge', 'degree', 'Bachelor', 'master'}

wc = WordCloud(
    width=1200, height=600,
    background_color='#1E1E2E',
    colormap='plasma',
    max_words=150,
    stopwords=custom_stopwords,
    collocations=False,
    prefer_horizontal=0.9,
).generate(skill_text)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Most In-Demand Skills — Word Cloud', fontweight='bold', fontsize=16, pad=15)
plt.tight_layout()
savefig(fig, '02_skills_wordcloud')
plt.show()

## 4. Top Skills by Role Category

In [ ]:
skills_with_role = skills_exploded.merge(
    df[['job_link', 'role_category', 'job_type']], on='job_link', how='left'
)

role_skill = (
    skills_with_role.groupby(['role_category', 'skill'])
    .size().reset_index(name='count')
)

top_roles = df['role_category'].value_counts().head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

palette_list = list(PALETTE.values())

for i, role in enumerate(top_roles):
    ax   = axes[i]
    data = role_skill[role_skill['role_category'] == role].nlargest(10, 'count')
    color = palette_list[i % len(palette_list)]
    
    bars = ax.barh(data['skill'][::-1].reset_index(drop=True),
                   data['count'][::-1].reset_index(drop=True),
                   color=color, alpha=0.85, edgecolor='none', height=0.7)
    
    ax.set_title(role, fontweight='bold', fontsize=11)
    ax.set_xlabel('Mentions')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Top 10 Skills by Role Category', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
savefig(fig, '02_skills_by_role')
plt.show()

## 5. Role × Skill Co-occurrence Heatmap

In [ ]:
top_skills_global = skill_freq.head(20)['skill'].tolist()

cooccurrence = (
    skills_with_role[
        skills_with_role['skill'].isin(top_skills_global) &
        skills_with_role['role_category'].isin(top_roles)
    ]
    .groupby(['role_category', 'skill'])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(
    cooccurrence, ax=ax, cmap='viridis',
    linewidths=0.5, linecolor='#1E1E2E',
    annot=True, fmt='d', annot_kws={'size': 8},
    cbar_kws={'label': 'Co-occurrences'}
)
ax.set_title('Role × Skill Co-occurrence Matrix (Top 20 Skills)', fontweight='bold', pad=15)
ax.set_xlabel('Skill')
ax.set_ylabel('Role Category')
ax.tick_params(axis='x', rotation=40, labelsize=8)
plt.tight_layout()
savefig(fig, '02_role_skill_cooccurrence')
plt.show()

## 6. Outsourcing & Remote Work Analysis

In [ ]:
remote_map = {
    'Remote'    : 'Remote',
    'Onsite'    : 'Onsite',
    'Hybrid'    : 'Hybrid',
    'Contract'  : 'Contract',
    'Part-time' : 'Part-time',
}
df['work_mode'] = df['job_type'].map(remote_map).fillna('Other')

mode_role = df.groupby(['work_mode', 'role_category']).size().unstack(fill_value=0)

fig = px.bar(
    mode_role.T.reset_index().melt(id_vars='role_category'),
    x='role_category', y='value', color='work_mode',
    barmode='group',
    title='Work Mode Distribution by Role Category',
    template=PLOTLY_TEMPLATE,
    labels={'value': 'Postings', 'role_category': 'Role', 'work_mode': 'Work Mode'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(xaxis_tickangle=-35, height=480)
save_plotly(fig, '02_work_mode_by_role')
fig.show()

## 7. Job Title Clustering (TF-IDF + KMeans)

In [ ]:
titles = df['job_title'].dropna().tolist()

tfidf = TfidfVectorizer(max_features=300, stop_words='english', ngram_range=(1, 2))
X     = tfidf.fit_transform(titles)

n_clusters = 8
km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
km.fit(X)

df_clust = df.dropna(subset=['job_title']).copy()
df_clust['cluster'] = km.labels_

# Print top terms per cluster
feature_names = tfidf.get_feature_names_out()
print('Top terms per cluster:')
for ci in range(n_clusters):
    center  = km.cluster_centers_[ci]
    top_idx = center.argsort()[::-1][:5]
    terms   = [feature_names[i] for i in top_idx]
    size    = (km.labels_ == ci).sum()
    print(f'  Cluster {ci} ({size:4d} jobs): {" | ".join(terms)}')

In [ ]:
# 2-D PCA visualisation
pca  = PCA(n_components=2, random_state=42)
X2d  = pca.fit_transform(X.toarray())

pca_df = pd.DataFrame(X2d, columns=['PC1','PC2'])
pca_df['cluster'] = km.labels_
pca_df['title']   = df_clust['job_title'].values

fig = px.scatter(
    pca_df, x='PC1', y='PC2', color=pca_df['cluster'].astype(str),
    hover_data={'title': True},
    title='Job Title Clusters (TF-IDF + KMeans, PCA Projection)',
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=px.colors.qualitative.Plotly,
    labels={'color': 'Cluster'}
)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(height=520)
save_plotly(fig, '02_job_title_clusters')
fig.show()

## 8. Interactive Skill Explorer

In [ ]:
top30 = skill_freq.head(30)

fig = px.treemap(
    top30,
    path=['skill'],
    values='count',
    title='Top 30 Skills — Treemap View',
    template=PLOTLY_TEMPLATE,
    color='count',
    color_continuous_scale='Plasma'
)
fig.update_layout(height=500)
save_plotly(fig, '02_skills_treemap')
fig.show()

---
## Key Findings

| Finding | Detail |
|---|---|
| Most demanded skill | `skill_freq.iloc[0]['skill']` |
| Largest role group | `df.role_category.value_counts().idxmax()` |
| Primary work mode | `df.work_mode.value_counts().idxmax()` |
| Outsourcing-relevant roles | Data Engineer, ML Engineer, Data Analyst |

➡️ **Next: Notebook 03 — Time Series Forecasting**